# Notebook 3: Run Offline Replay Experiments (E1-E6)

This notebook triggers the main offline experimentation pipeline. It simulates thousands of passengers making decisions under different policy configurations over a specified time period.

**Process:**
1. **Data Collection**: The `ExperimentRunner` first collects a series of real-time airport snapshots.
2. **Simulation**: For each snapshot, it simulates a batch of passengers, each with different profiles and scenarios (Q1-Q4).
3. **Policy Evaluation**: For each simulated passenger, it evaluates all defined policies (Baseline, Hybrid-Mean, Hybrid-CVaR, etc.) to get a recommended route.
4. **Logging**: It logs the recommended route, the simulated user's choice (with epsilon-greedy acceptance), the realized journey time, and other metadata to the `policy_evaluation_log` table.
5. **Aggregation**: Finally, it computes and stores summary metrics (APT, CVaR, Miss Rate) for each policy/scenario pair in the `experiment_metrics` table.

⚠️ **Warning**: A full experiment run can take a significant amount of time (e.g., 30-60 minutes) depending on the configuration, as it involves numerous API calls and computations.

The output of this notebook is a unique **`experiment_id`** (UUID), which you will need for the analysis in the next notebook.

In [ ]:
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.db import get_engine, FeatureStore
from src.pipeline.engine import AirportOptimizationEngine
from src.pipeline.experiments import ExperimentConfig, ExperimentRunner

### 1. Configure the Experiment

You can adjust the parameters of the experiment here. For a quick test, use a short duration and a small number of users. For a full-scale analysis, use the default or larger values.

In [ ]:
# --- DEMO CONFIGURATION (for a quick run) ---
exp_config = ExperimentConfig(
    duration_minutes=15,      # Short duration for the demo
    interval_minutes=5,       # Snapshot every 5 minutes
    users_per_snapshot=3,     # Small number of users
    epsilon=0.2               # 20% chance user ignores recommendation
)

# --- FULL-SCALE CONFIGURATION (for robust analysis) ---
# exp_config = ExperimentConfig(
#     duration_minutes=180,     # 3 hours
#     interval_minutes=5,       # Snapshot every 5 minutes
#     users_per_snapshot=10,    # 10 users per snapshot
#     epsilon=0.2
# )

print("Using Experiment Configuration:")
print(f"- Duration: {exp_config.duration_minutes} minutes")
print(f"- Interval: {exp_config.interval_minutes} minutes")
print(f"- Users per Snapshot: {exp_config.users_per_snapshot}")
print(f"- Epsilon: {exp_config.epsilon}")

### 2. Initialize the Experiment Runner

In [ ]:
try:
    db_engine = get_engine()
    feature_store = FeatureStore(db_engine)
    app_engine = AirportOptimizationEngine(db_engine, feature_store)
    runner = ExperimentRunner(app_engine, db_engine)
    print("✅ ExperimentRunner initialized successfully.")
except Exception as e:
    print(f"❌ Initialization failed: {e}")

### 3. Run the Experiment

This cell will execute the entire experiment. Please be patient.

In [ ]:
experiment_id = None
try:
    print("Starting experiment run... This will take some time.")
    experiment_id = runner.run(exp_config)
except Exception as e:
    print(f"❌ Experiment run failed: {e}")

### 4. Capture the Experiment ID

The experiment is complete. The unique ID below is crucial for analyzing the results in the next notebook. **Copy this ID.**

In [ ]:
if experiment_id:
    print("="*50)
    print("  EXPERIMENT COMPLETE  ")
    print("="*50)
    print(f"\nExperiment ID: {experiment_id}")
    print("\n📋 Please copy the ID above and paste it into the next notebook (04_statistical_analysis.ipynb).")
else:
    print("\nExperiment did not produce an ID. Please check the logs for errors.")